<h1 align=center><i> Proyecto Final Semillero: Mesa de Ayuda IA para el Departamento de Ventas de Patito S.A.</i></h1>

<h3 align=center><i>Agentes Colaborativos con LangChain + Google Gemini 🤖</i></h3>

---
## <span style="color: #1a365d;"> Grupo: Modo Avión ✈️</span>

### 👥 **Integrantes:**
  * Alvarado María Gisselie
  * Huanca Ashley Briggitte
  * Yugsan Leonardo Mateo

### 1.  Objetivo
Desarrollar un chatbot multiagente en **JupyterLab** que apoye al departamento de Ventas mediante consultas inteligentes, recuperación de información (RAG) y herramientas de acción para automatizar tareas comerciales.

### 2. Arquitectura del Sistema

| # | Componente | Función | Implementación |
|:--|:-------------|:-----------|:-----------------|
| 1 | **Catálogo** | Consulta productos y precios. | RAG + ChromaDB |
| 2 | **Políticas** | Gestiona descuentos y créditos. | RAG + ChromaDB |
| 3 | **CRM** | Asiste el proceso de ventas. | RAG |
| 4 | **Multimodal** | Analiza imágenes. | `gemini-3.1-flash-lite` |
| 5 | **Registro** | Registra oportunidades de venta. | `@tool` + Function Calling |
| 6 | **Orquestador** | Coordina los agentes especializados. | LangGraph + LangChain |
| 7 | **Observabilidad** | Monitorea trazas y métricas. | Arize Phoenix |
| 8 | **Interfaz** | Chat interactivo en JupyterLab. | `ipywidgets` + `threading` |

> **Tecnologías principales:** Python, Google Gemini, LangChain, LangGraph, ChromaDB, Arize Phoenix, JupyterLab.

### 3.  Aspectos Destacados
- Recuperación semántica con RAG.
- Control de alucinaciones mediante *prompts* y fuentes.
- Credenciales protegidas con `.env`.
- Interfaz no bloqueante mediante `threading`.


## 🛠️ 1. Instalacción de dependencias obligatorias

- Se instalan las dependencias para orquestación (langchain), modelos de lenguaje e integración con Google Gemini (langchain-google-genai), base de datos vectorial (chromadb, langchain-chroma) y división de texto (langchain-text-splitters).

- Se incluye pillow para la manipulación de imágenes, pandas, e ipywidgets para la interfaz gráfica. Para el monitoreo y trazabilidad, se instalan arize-phoenix y openinference-instrumentation-langchain.

In [1]:
# LangChain + Google Gemini + ChromaDB
!pip install -q langchain langchain-google-genai langchain-community langchain-chroma chromadb langchain-text-splitters
# Utilidades
!pip install -q pillow pandas ipywidgets python-dotenv
# Observabilidad con Phoenix
!pip install -q arize-phoenix openinference-instrumentation-langchain
print("Instalación completa.")


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Instalación completa.



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import sys

!{sys.executable} -m pip install mypy_extensions


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 🔌 2. Conexion a Google Gemini

Configuramos **un mismo proveedor (Gemini)** para todo:

Modelos configurados:

- LLM: `gemini-3.1-flash-lite` para razonamiento, procesamiento del lenguaje natural y capacidades multimodales (visión).

- Embeddings: `gemini-embedding-2-preview` (o text-embedding-004) para la conversión de texto a vectores densos.

Prueba de conexión: Se realiza un ping simple invocar `llm.invoke()` para asegurar la correcta lectura de la API key antes de proceder.

In [6]:
import os, getpass
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings

load_dotenv()
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")

MODELO_LLM = "gemini-3.1-flash-lite"
MODELO_EMBEDDING = "gemini-embedding-2-preview"

llm = ChatGoogleGenerativeAI(model=MODELO_LLM, temperature=0)
embeddings = GoogleGenerativeAIEmbeddings(model=MODELO_EMBEDDING)

# Ping: si esto imprime el saludo, estas conectado
print(llm.invoke("Responde unicamente: 'Gemini conectado.' y nada mas.").content)

[{'type': 'text', 'text': 'Gemini conectado.', 'extras': {'signature': 'EjQKMgERTTIPAM1ANZ3NKZxHUMmKFd2y+XgZ3hKW94kZS1PVC70pN7y7hRj2wmTx6yvWYSb+'}}]


 ## Arquitectura de la solución


La solución implementa una arquitectura **RAG (Retrieval-Augmented Generation)** basada en **agentes especializados** coordinados por un **orquestador**. El sistema utiliza **Google Gemini** como modelo de lenguaje, **Chroma** como base vectorial para la recuperación de información y **Phoenix** para la observabilidad de las ejecuciones.

### Arquitectura General

```text
                           Usuario
                               │
                               ▼
                   Interfaz del Asistente
                     (JupyterLab - ipywidgets)
                               │
                               ▼
                     Agente Orquestador
                  (LangChain + create_agent)
                               │
        ┌──────────────────────┼──────────────────────┐
        │                      │                      │
        ▼                      ▼                      ▼
 Agente Catálogo      Agente Políticas        Agente CRM
   y Precios           Comerciales       Proceso de Ventas
        │                      │                      │
        ▼                      ▼                      ▼
Retriever Catálogo   Retriever Políticas   Retriever CRM
        │                      │                      │
        └──────────────┬───────────────┬──────────────┘
                       ▼
            Bases Vectoriales (Chroma)
                       │
                       ▼
      GoogleGenerativeAIEmbeddings
                       │
                       ▼
      Documentos de conocimiento (.txt)
                       │
                       ▼
             Google Gemini (LLM)
                       │
                       ▼
          Respuesta al usuario

        ┌──────────────────────────────┐
        │ Agente de Acción             │
        │ registrar_oportunidad()      │
        │                              │
        ▼                              │
registro_oportunidades.txt             │
        └──────────────────────────────┘

        ┌──────────────────────────────┐
        │ Agente Multimodal            │
        │ analizar_imagen()            │
        └──────────────────────────────┘
                     |
                     ▼
              Observabilidad
                  Phoenix
```

## Componentes de la Arquitectura

1.  **Interfaz del Asistente**: Permite al usuario interactuar con el sistema mediante una interfaz desarrollada en JupyterLab utilizando `ipywidgets`. Desde esta interfaz se envían las consultas al orquestador.
2.  **Agente Orquestador**: Analiza la intención de la consulta y determina qué agente especializado o herramienta debe intervenir. Además, mantiene el flujo de conversación y coordina el uso de las herramientas disponibles.
3.  **Agentes Especializados:**
| # | Agente | Descripción |
|---|---|---|
| 1 | **Agente Catálogo y Precios:** | Responde consultas sobre productos, precios, disponibilidad y características técnicas. |
| 2 | **Agente de Políticas Comerciales:** | Responde preguntas relacionadas con descuentos, garantías, devoluciones, crédito y anticipos. |
| 3 | **Agente de Proceso de Ventas y CRM:** | Responde consultas sobre el proceso comercial y el uso del CRM. |
4.  **Agente Multimodal**: Analiza imágenes utilizando las capacidades multimodales de `Google Gemini` para extraer información relevante.
5.  **Agente de Acción**: Ejecuta operaciones sobre el sistema, como el registro de oportunidades comerciales, validando previamente que todos los datos obligatorios estén presentes antes de almacenarlos.
6.  **Bases Vectoriales**: Cada documento de conocimiento se divide en fragmentos **(chunks)**, se transforma en `embeddings` mediante `GoogleGenerativeAIEmbeddings` y se almacena en una colección independiente de `Chroma` para facilitar la recuperación de información.
6.  **Google Gemini**: Modelo de lenguaje utilizado para generar respuestas basadas en el contexto recuperado, así como para soportar el funcionamiento del orquestador y del agente multimodal.
7.  **Phoenix**: Herramienta de observabilidad encargada de registrar las trazas de ejecución del sistema, incluyendo llamadas al modelo, retrievers, herramientas utilizadas, tiempos de respuesta y consumo de tokens.

## Flujo de Funcionamiento

```text
Usuario
   │
   ▼
Interfaz del Asistente
   │
   ▼
Agente Orquestador
   │
   ├──► Agente Catálogo
   ├──► Agente Políticas
   ├──► Agente CRM
   ├──► Agente Multimodal
   └──► Agente de Acción
            │
            ▼
      Google Gemini / Chroma
            │
            ▼
      Respuesta al Usuario
            │
            ▼
      Phoenix registra la traza
```
**Detalle**
1. El usuario envía una consulta desde la interfaz.
2. El orquestador identifica la intención de la solicitud.
3. Se selecciona el agente especializado o la herramienta correspondiente.
4. Si la consulta requiere información documental, el retriever recupera los fragmentos más relevantes desde `Chroma`.
5. `Google Gemini` genera la respuesta utilizando el contexto recuperado.
6. Si la solicitud implica una acción, el Agente de Acción ejecuta la operación correspondiente.
7. `Phoenix` registra automáticamente todo el proceso para su posterior análisis.

## 📊 3. Phoenix 
Phoenix se inicializa **antes** de los imports de LangChain. ¿Por qué?

La instrumentación funciona "parchando" la librería de LangChain en tiempo de import. Si LangChain ya fue importada y usada, los primeros calls no quedan capturados. Por eso este bloque va **al inicio**.

Tres pasos:

1. **`launch_app()`** — levanta el servidor Phoenix en `localhost:6006`.
2. **`register()`** — crea el tracer apuntando al servidor con un nombre de proyecto.
3. **`LangChainInstrumentor().instrument()`** — a partir de aquí, **cada `invoke` se captura solo**.

> ⚠️ **Si re-ejecutas esta celda** y ves un error de puerto ocupado o de doble instrumentación, reinicia el kernel (`Kernel > Restart`).

In [7]:
import phoenix as px
from phoenix.otel import register
from openinference.instrumentation.langchain import LangChainInstrumentor

# 1. Levanta el servidor local de Phoenix
session = px.launch_app()

# 2. Tracer apuntando al servidor local, con el nombre del proyecto
tracer_provider = register(project_name="Proyecto-Final")

# 3. Instrumenta LangChain - guard para que re-ejecutar la celda no rompa
try:
    LangChainInstrumentor().instrument(tracer_provider=tracer_provider)
except Exception as e:
    print(f"(Instrumentacion ya activa: {e})")

print(f"\nPhoenix UI:  {session.url}")
print("Abrela en otra pestana. A partir de aqui cada invoke al agente se ve ahi.")

C:\Users\USUARIO\AppData\Local\Programs\Python\Python311\Lib\contextlib.py:144: SAWarning: Skipped unsupported reflection of expression-based index ix_cumulative_llm_token_count_total
  next(self.gen)
C:\Users\USUARIO\AppData\Local\Programs\Python\Python311\Lib\contextlib.py:144: SAWarning: Skipped unsupported reflection of expression-based index ix_latency
  next(self.gen)
C:\Users\USUARIO\AppData\Local\Programs\Python\Python311\Lib\contextlib.py:144: SAWarning: Skipped unsupported reflection of expression-based index ix_spans_session_id
  next(self.gen)
C:\Users\USUARIO\AppData\Local\Programs\Python\Python311\Lib\site-packages\pydantic\json_schema.py:2463: PydanticJsonSchemaWarning: Default value <phoenix.db.types.db_helper_types.Undefined object at 0x00000279063C4850> is not JSON serializable; excluding default from JSON schema [non-serializable-default]
  warnings.warn(message, PydanticJsonSchemaWarning)
boto3 is installed but aioboto3 is not. To use AWS Bedrock models in Playgroun

🌍 To view the Phoenix app in your browser, visit http://localhost:6006/
📖 For more information on how to use Phoenix, check out https://arize.com/docs/phoenix
OpenTelemetry Tracing Details
|  Phoenix Project: Proyecto-Final
|  Span Processor: SimpleSpanProcessor
|  Collector Endpoint: localhost:4317
|  Transport: gRPC
|  Transport Headers: {}
|  
|  Using a default SpanProcessor. `add_span_processor` will overwrite this default.
|  
|  WARNING: It is strongly advised to use a BatchSpanProcessor in production environments.
|  
|  `register` has set this TracerProvider as the global OpenTelemetry default.
|  To disable this behavior, call `register` with `set_global_tracer_provider=False`.


Phoenix UI:  http://localhost:6006/
Abrela en otra pestana. A partir de aqui cada invoke al agente se ve ahi.


## 📄 4. Base de conocimiento - Base documental
El agente de conocimiento no debe inventar las reglas: debe responder a partir del documento oficial. Se realiza la lectura de tres archivos planos (.txt) independientes alojados en el directorio local:

`01_Catalogo_Productos_Precios.txt`

`02_Politicas_Comerciales_Descuentos_Credito.txt`

`03_Proceso_Ventas_CRM.txt`

In [8]:
from pathlib import Path

Documentos = Path("documentos")

catalogo = (Documentos / "01_Catalogo_Productos_Precios.txt").read_text(encoding="utf-8")
politicas = (Documentos / "02_Politicas_Comerciales_Descuentos_Credito.txt").read_text(encoding="utf-8")
crm = (Documentos / "03_Proceso_Ventas_CRM.txt").read_text(encoding="utf-8")

print(f"Catálogo: {len(catalogo):,} caracteres")
print(f"Políticas: {len(politicas):,} caracteres")
print(f"CRM: {len(crm):,} caracteres")

Catálogo: 1,104 caracteres
Políticas: 1,474 caracteres
CRM: 1,168 caracteres


## ✂️ 5. Chunking + Embeddings + Chroma 

Cortamos la politica en **chunks** (un chunk por seccion numerada), convertimos cada chunk en un **vector** con los embeddings de Gemini y los guardamos en **ChromaDB**, que sabe buscar por similitud.
1.  Dado que la documentación institucional tiene títulos estructurados por números (ej. 1., 2.), se utiliza una expresión regular `(r"^\d+\.\s")` para dividir los archivos por artículos completos en lugar de cortes por tamaño fijo.
2.  Se crean tres colecciones separadas en ChromaDB `(patito_catalogo, patito_politicas, patito_crm)`.
3.  Se configuran buscadores de k-vecinos más cercanos ($k=3$) con metadatos asociados para evitar que el contexto de un área contamine el razonamiento de otra.

>

In [9]:
import re
from langchain_chroma import Chroma

def chunkear_por_secciones(texto):
    """ Divide el documento usando encabezados numerados: 1., 2., 3.  """
    cabeceras = list(re.finditer(r"^\d+\.\s", texto, flags=re.MULTILINE))
    chunks = []
    for i, cabecera in enumerate(cabeceras):
        inicio = cabecera.start()
        fin = cabeceras[i + 1].start() if i + 1 < len(cabeceras) else len(texto)
        chunks.append(texto[inicio:fin].strip())
    return chunks
    
# Generar los chunks
chunks_catalogo = chunkear_por_secciones(catalogo)
chunks_politicas = chunkear_por_secciones(politicas)
chunks_crm = chunkear_por_secciones(crm)


# Mostrar un resumen
print("===== RESUMEN DEL CHUNKING =====")

print(f"\nCatálogo: {len(chunks_catalogo)} chunks")
for i, chunk in enumerate(chunks_catalogo, start=1):
    print(f"Chunk {i}: {chunk.splitlines()[0]}")

print(f"\nPolíticas: {len(chunks_politicas)} chunks")
for i, chunk in enumerate(chunks_politicas, start=1):
    print(f"Chunk {i}: {chunk.splitlines()[0]}")

print(f"\nCRM: {len(chunks_crm)} chunks")
for i, chunk in enumerate(chunks_crm, start=1):
    print(f"Chunk {i}: {chunk.splitlines()[0]}")

# VECTOR STORE: CATÁLOGO
vectorstore_catalogo = Chroma.from_texts(
    texts=chunks_catalogo,
    embedding=embeddings,
    metadatas=[{"seccion": i,"fuente": "01_Catalogo_Productos_Precios.txt"} for i in range(len(chunks_catalogo))],
    collection_name="patito_catalogo"
)
# VECTOR : POLÍTICAS
vectorstore_politicas = Chroma.from_texts(
    texts=chunks_politicas,
    embedding=embeddings,
    metadatas=[{"seccion": i,"fuente": "02_Politicas_Comerciales_Descuentos_Credito.txt"}for i in range(len(chunks_politicas))],
    collection_name="patito_politicas"
)
# VECTOR STORE: CRM
vectorstore_crm = Chroma.from_texts(
    texts=chunks_crm,
    embedding=embeddings,
    metadatas=[{"seccion": i,"fuente": "03_Proceso_Ventas_CRM.txt"}for i in range(len(chunks_crm))],
    collection_name="patito_crm"
)
retriever_catalogo = vectorstore_catalogo.as_retriever(search_kwargs={"k": 3})
retriever_politicas = vectorstore_politicas.as_retriever(search_kwargs={"k": 3})
retriever_crm = vectorstore_crm.as_retriever(search_kwargs={"k": 3})
print("Base de conocimiento embebida en Chroma con embeddings de Gemini.")


===== RESUMEN DEL CHUNKING =====

Catálogo: 4 chunks
Chunk 1: 1. LÍNEA PATITO PRO
Chunk 2: 2. LÍNEA PATITO LITE
Chunk 3: 3. ACCESORIOS
Chunk 4: 4. NOTAS

Políticas: 5 chunks
Chunk 1: 1. DESCUENTOS (NIVELES DE AUTORIZACIÓN)
Chunk 2: 2. CONDICIONES DE CRÉDITO
Chunk 3: 3. GARANTÍAS
Chunk 4: 4. DEVOLUCIONES
Chunk 5: 5. ANTICIPOS

CRM: 5 chunks
Chunk 1: 1. ETAPAS DEL EMBUDO (CRM)
Chunk 2: 2. REGISTRO EN EL CRM
Chunk 3: 3. REQUISITOS PARA MARCAR UNA OPORTUNIDAD COMO "GANADA"
Chunk 4: 4. POSVENTA
Chunk 5: 5. BUENAS PRÁCTICAS
Base de conocimiento embebida en Chroma con embeddings de Gemini.


In [ ]:
# PRUEBAS DE RECUPERACIÓN DE INFORMACIÓN
def probar_retriever(nombre, retriever, pregunta):
    print("=" * 70)
    print(nombre)
    print("=" * 70)
    print(f"Pregunta: {pregunta}\n")
    documentos = retriever.invoke(pregunta)
    print(f"Documentos recuperados: {len(documentos)}\n")
    for i, doc in enumerate(documentos, start=1):
        print(f"--- Resultado {i} ---")
        print(f"Fuente: {doc.metadata}")
        print(doc.page_content[:400])
        print()
# Prueba catálogo
probar_retriever("RETRIEVER CATÁLOGO", retriever_catalogo, "¿Cuál es el precio del Patito Pro 2026?")
# Prueba políticas
probar_retriever("RETRIEVER POLÍTICAS", retriever_politicas, "¿Qué descuento puede autorizar directamente un vendedor?")
# Prueba CRM
probar_retriever("RETRIEVER CRM", retriever_crm,"¿Qué requisitos se necesitan para marcar una oportunidad como ganada?")

## 🧠 6. Agentes Especializados (Módulo RAG + Multimodal)

### 📗 6.1. Agente Catálogo y Precios
Se crea el agente de **catálogo y precios** mediante una función RAG que recibe una pregunta, consulta la base vectorial correspondiente y utiliza *Gemini* para responder únicamente con la información encontrada. 
> Responde dudas exclusivas sobre productos, spec sheets, precios y stock disponible. 

*En caso de no encontrar datos en su contexto vectorial, responde una frase rígida previa para evitar alucinaciones.*

In [10]:
PROMPT_CATALOGO = """Eres el Agente de Catálogo y Precios de PATITO S.A.
Respondes consultas sobre:
- Productos.
- Precios.
- Disponibilidad.
- Características técnicas.

Reglas:
- Responde únicamente con base en el CONTEXTO.
- No inventes información.
- Si la respuesta no aparece en el contexto responde exactamente:
"No tengo esa informacio en el catálogo actual."
- Sé breve y directo.
"""

def responder_catalogo(pregunta: str) -> str:
    """Pipeline RAG del catálogo."""
    docs = retriever_catalogo.invoke(pregunta)
    contexto = "\n\n---\n\n".join(
        doc.page_content
        for doc in docs
    )
    respuesta = llm.invoke([
        {"role": "system","content": PROMPT_CATALOGO},
        {"role": "user","content": f"CONTEXTO:\n{contexto}\n\nPREGUNTA: {pregunta}"}
    ])
    return respuesta.content

# Prueba
print(responder_catalogo("Cual es el monto maximo de un almuerzo de trabajo?"))

[{'type': 'text', 'text': 'No tengo esa informacio en el catálogo actual.', 'extras': {'signature': 'EjQKMgERTTIPiCM28IRDdxQKkfHh9Ql4NqlnWwjtyIA4i/FumFCu8Ss7ZLMaht0gWOLQnER3'}}]


### ⚖️ 6.2. Agente de Políticas Comerciales
Se implementa la función RAG para el agente de **Políticas Comerciales**, aplicando un prompt estricto que restringe la respuesta a las normas explícitas de la empresa para evitar alucinaciones. 
> Responde sobre descuentos autorizados, condiciones de crédito, devoluciones y garantías, con reglas explícitas para restringir respuestas no autorizadas.

In [11]:
prompt_politicas = ("""Eres el Agente de Políticas Comerciales de PATITO S.A.

Tu función es responder únicamente consultas relacionadas con:
- descuentos;
- niveles de autorización;
- condiciones de crédito;
- garantías;
- devoluciones;
- anticipos.

Reglas:
- Utiliza exclusivamente el contexto proporcionado.
- No inventes políticas ni condiciones comerciales.
- Si la información no existe en las políticas responde:
  "No dispongo de esa información en las políticas comerciales actuales."
- Responde de forma clara y profesional.
"""
)

def responder_politicas(pregunta:str) -> str:
    """Pipeline RAG del catálogo"""
    docs = retriever_politicas.invoke(pregunta)
    contexto = "\n\n---\n\n".join(
        doc.page_content
        for doc in docs
    )
    
    respuesta = llm.invoke([
        {"role": "system","content": prompt_politicas},
        {"role": "user","content": f"CONTEXTO:\n{contexto}\n\nPREGUNTA: {pregunta}"}
    ])
    return respuesta.content
# Prueba descuento permitido
print(responder_politicas("¿Qué porcentaje de descuento puede autorizar directamente un vendedor?"))
# Prueba crédito
print(responder_politicas("¿Qué requisitos existen para vender a crédito?"))
# Prueba control de alucinación
print (responder_politicas("¿Patito S.A. ofrece descuentos del 80%?"))

[{'type': 'text', 'text': 'Un vendedor puede autorizar directamente un descuento de hasta el 10%, sin necesidad de aprobación adicional.', 'extras': {'signature': 'EjQKMgERTTIPopMADYxgpc4M9WCirPIMwdjfUDPD2R00N4ltmPcDkwfYpVHdxCXBiMYyPPWD'}}]
[{'type': 'text', 'text': 'Para realizar ventas a crédito en PATITO S.A., se deben cumplir los siguientes requisitos:\n\n*   **Solicitud de crédito:** Presentar la solicitud formal.\n*   **Documentación:** Entregar los documentos de la empresa.\n*   **Evaluación:** La aprobación por parte del área financiera.\n\nAdicionalmente, tenga en cuenta que para clientes nuevos, la primera compra suele ser de contado y el crédito se evalúa tras la primera operación y un análisis de crédito. Asimismo, las ventas a crédito sin una línea aprobada requieren autorización de la gerencia.', 'extras': {'signature': 'EjQKMgERTTIPXvJGz/komGU15mgnMOPuANrjsbywZWXu8LKWFJkd4nWCvGPy3pXeysMfjJ8L'}}]
[{'type': 'text', 'text': 'No, los descuentos superiores al 30% no están per

### 💸 6.3. Agente Proceso de Ventas y CRM
Se configura el agente de **CRM** mediante un pipeline RAG encargado de resolver dudas operativas sobre el pipeline de ventas y las etapas del proceso comercial.
> Gestiona el conocimiento sobre etapas del embudo de ventas, buenas prácticas de seguimiento y condiciones obligatorias para cerrar ventas como "ganadas".

In [12]:
prompt_crm = ("""Eres el Agente de Proceso de Ventas y CRM de PATITO S.A.
Tu función es responder únicamente consultas relacionadas con:
- etapas del proceso comercial;
- registro de oportunidades;
- requisitos del CRM;
- cierre de ventas;
- seguimiento posventa.

Reglas:
- Utiliza solamente la información del contexto recuperado.
- No inventes procesos, campos del CRM ni requisitos.
- Si la información no aparece en el manual responde:
  "No dispongo de esa información en el proceso de ventas y CRM actual."
- Mantén respuestas claras y profesionales.
"""
)
def responder_crm(pregunta:str) -> str:
    """Pipeline RAG del catálogo"""
    docs = retriever_crm.invoke(pregunta)
    contexto = "\n\n---\n\n".join(
        doc.page_content
        for doc in docs
    )
    
    respuesta = llm.invoke([
        {"role": "system","content": prompt_crm},
        {"role": "user","content": f"CONTEXTO:\n{contexto}\n\nPREGUNTA: {pregunta}"}
    ])
    return respuesta.content

# Prueba 1 etapas del embudo
print(responder_crm("¿Cuáles son las etapas del embudo de ventas?"))
#Prueba 2 oportunidad ganada
print(responder_crm("¿Qué requisitos se necesitan para marcar una oportunidad como ganada?"))
#Prueba 3 pregunta fuera de dominio
print(responder_crm("¿Cuál es el precio del Patito Pro 2026?"))

[{'type': 'text', 'text': 'Las etapas del embudo de ventas en el CRM de PATITO S.A. son las siguientes:\n\n1. Prospecto\n2. Contacto\n3. Calificación\n4. Propuesta/Cotización\n5. Negociación\n6. Cierre (Ganada o Perdida)', 'extras': {'signature': 'EjQKMgERTTIPR+CYLtI4KB5q/m1dxMxzYJlUKwVsI9FqoFu7LGMSUVcskDJrVEEno16lnAYO'}}]
[{'type': 'text', 'text': 'Para marcar una oportunidad como "ganada" en el CRM, es obligatorio registrar los siguientes requisitos:\n\n*   Orden de compra o contrato firmado por el cliente.\n*   Datos de facturación completos del cliente.\n*   Productos, cantidades y precios finales (incluyendo el descuento aplicado y su autorización).\n*   Condición de pago (contado o crédito) y plazo acordado.\n*   Monto total de la venta y fecha de cierre.\n*   Fecha de entrega comprometida.', 'extras': {'signature': 'EjQKMgERTTIPNsWg8HCVDqsohU0yv8xLx/CWjBkHELCd0h8pjMYESMGQovxUPnsYAIsPhYT+'}}]
[{'type': 'text', 'text': 'No dispongo de esa información en el proceso de ventas y CRM 

### 🖼️ 6.4. Agente Multimodal de imagen

**Gemini es multimodal**: puede recibir una imagen y leerla. 
Genera sintéticamente una imagen local *(patito_pro.png)* usando la librería `Pillow` para simular una ficha técnica de un producto con especificaciones visuales.

> Utiliza las capacidades de visión de Gemini para tomar imágenes (en el ejemplo, creadas sintéticamente con `Pillow` y codificadas en `base64`) y extraer metadatos estructurados como precios, RAM y disponibilidad

In [13]:
from PIL import Image, ImageDraw
import base64

def crear_ficha_producto_demo(ruta="patito_pro.png"):
    """Genera una imagen simple de un producto de PATITO S.A."""

    img = Image.new("RGB", (500, 350), "white")
    d = ImageDraw.Draw(img)

    lineas = [
        "PATITO S.A.",
        "CATÁLOGO DE PRODUCTOS",
        "--------------------------------",
        "Producto: Patito Pro 2026",
        "Precio: USD 1299",
        "Disponibilidad: EN STOCK",
        "Garantía: 12 meses",
        "RAM: 16 GB",
        "SSD: 512 GB"
    ]
    y = 20
    for linea in lineas:
        d.text((20, y), linea, fill="black")
        y += 32

    img.save(ruta)

    return ruta

ruta_imagen = crear_ficha_producto_demo()

print("Imagen creada:", ruta_imagen)

Imagen creada: patito_pro.png


In [14]:
from langchain_core.messages import HumanMessage
import base64

def analizar_producto(ruta_imagen: str) -> str:
    """Analiza una imagen de un producto de PATITO S.A."""

    try:
        with open(ruta_imagen, "rb") as f:
            b64 = base64.b64encode(f.read()).decode()
    except FileNotFoundError:
        return f"No se encontró la imagen '{ruta_imagen}'."

    prompt = (
        "Analiza esta imagen de un producto de PATITO S.A. "
        "Extrae y devuelve en líneas separadas: "
        "producto, precio, disponibilidad, garantía y características visibles. "
        "Si algún dato no aparece escribe 'No visible'. "
        "No inventes información."
    )

    msg = HumanMessage(content=[
        {"type": "text", "text": prompt},
        {"type": "image_url", "image_url": f"data:image/png;base64,{b64}"}
    ])

    return llm.invoke([msg]).content

## ✍️ 7. Agente de Acción (Registro) - Sistema de control

Se construye la herramienta *@tool registrar_oportunidad*, la cual valida campos requeridos, exige bandera de confirmación, aplica reglas de descuento y guarda las oportunidades en un archivo local .txt.

Función `@tool` (`registrar_oportunidad`):
   - **Validación de campos**: Verifica la presencia de datos mínimos obligatorios (cliente, contacto, producto, cantidad, precio, descuento, condición de pago, monto).
   - **Reglas de negocio**: Si el descuento supera el $10\%$, exige un campo adicional de `autorizacion_descuento`.
   - **Confirmación explícita**: Si no se marca `confirmar=True`, retorna solo un borrador en texto y solicita la confirmación antes de escribir en el archivo plano `registro_oportunidades.txt`.

In [15]:
from langchain.tools import tool
from datetime import datetime
from pathlib import Path

REGISTRO_PATH = "registro_oportunidades.txt"
CAMPOS_OBLIGATORIOS = ["cliente", "contacto", "producto", "cantidad", "precio_unitario", "descuento_aplicado", "condicion_pago", "monto_total",]

def _siguiente_id():
    if not Path(REGISTRO_PATH).exists():
        return "OP-0001"
    n = sum(1 for l in open(REGISTRO_PATH, encoding="utf-8") if l.strip())
    return f"OP-{n + 1:04d}"

@tool
def registrar_oportunidad(cliente: str = "", contacto: str = "", producto: str = "", cantidad: int = 0, precio_unitario: float = 0.0,
                          descuento_aplicado: float = 0.0, autorizacion_descuento: str = "", condicion_pago: str = "", monto_total: float = 0.0,
                          confirmar: bool = False,) -> str:
    """ Registra una oportunidad comercial en un archivo de texto. Requiere:cliente, contacto, producto, cantidad, precio_unitario,
    descuento_aplicado, condicion_pago y monto_total. Si el descuento supera 10%, requiere autorizacion_descuento. Si falta información, 
    no registra y devuelve los datos faltantes. Si confirmar=False, solo muestra el resumen y solicita confirmación. """
    datos = {"cliente": cliente, "contacto": contacto, "producto": producto, "cantidad": cantidad, "precio_unitario": precio_unitario,
             "descuento_aplicado": descuento_aplicado, "autorizacion_descuento": autorizacion_descuento, "condicion_pago": condicion_pago,
             "monto_total": monto_total}

    faltantes = [
        k for k in CAMPOS_OBLIGATORIOS
        if not str(datos[k]).strip() or (k in {"cantidad", "precio_unitario", "monto_total"} and float(datos[k]) <= 0)
    ]

    if float(descuento_aplicado) > 10 and not str(autorizacion_descuento).strip():
        faltantes.append("autorizacion_descuento")

    faltantes = sorted(set(faltantes))
    if faltantes:
        return "No se registró la oportunidad. Faltan datos obligatorios: " + ", ".join(faltantes) + "."

    resumen = (
        "Resumen de la oportunidad:\n" 
        f"- Cliente: {cliente}\n"
        f"- Contacto: {contacto}\n"
        f"- Producto: {producto}\n"
        f"- Cantidad: {cantidad}\n"
        f"- Precio unitario: USD {float(precio_unitario):.2f}\n"
        f"- Descuento aplicado: {float(descuento_aplicado):.2f}%\n"
        f"- Autorización descuento: {autorizacion_descuento or 'No aplica'}\n"
        f"- Condición de pago: {condicion_pago}\n"
        f"- Monto total: USD {float(monto_total):.2f}\n"
    )

    if not confirmar:
        return resumen + "\nConfirma la operación invocando nuevamente la tool con confirmar=True."

    rid = _siguiente_id()
    ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    registro = (
    f"ID: {rid}\n"
    f"Fecha: {ts}\n"
    f"Cliente: {cliente}\n"
    f"Producto: {producto}\n"
    f"Cantidad: {cantidad}\n"
    f"Monto estimado: USD {float(monto_total)}\n"
    f"Etapa: Prospecto\n"
    f"Estado: Abierta\n")
    
    with open(REGISTRO_PATH, "a", encoding="utf-8") as f:
        f.write(registro + "------------------------------\n")
    
    return registro

# Prueba 1: faltan datos
print(registrar_oportunidad.invoke({"cliente": "Comercial ABC", "producto": "Patito Pro 2026", "cantidad": 10}))
# Prueba 2: datos completos, pero sin confirmar
print(registrar_oportunidad.invoke({"cliente": "Comercial ABC", "contacto": "María López", "producto": "Patito Pro 2026", "cantidad": 10, "precio_unitario": 1299, "descuento_aplicado": 8, "condicion_pago": "Contado", "monto_total": 11990}))
# Prueba 3: confirmación y registro
print(registrar_oportunidad.invoke({"cliente": "Comercial ABC", "contacto": "María López", "producto": "Patito Pro 2026", "cantidad": 10, "precio_unitario": 1299, "descuento_aplicado": 8, "condicion_pago": "Contado", "monto_total": 11990, "confirmar": True}))

No se registró la oportunidad. Faltan datos obligatorios: condicion_pago, contacto, monto_total, precio_unitario.
Resumen de la oportunidad:
- Cliente: Comercial ABC
- Contacto: María López
- Producto: Patito Pro 2026
- Cantidad: 10
- Precio unitario: USD 1299.00
- Descuento aplicado: 8.00%
- Autorización descuento: No aplica
- Condición de pago: Contado
- Monto total: USD 11990.00

Confirma la operación invocando nuevamente la tool con confirmar=True.
ID: OP-0028
Fecha: 2026-07-26 21:47:48
Cliente: Comercial ABC
Producto: Patito Pro 2026
Cantidad: 10
Monto estimado: USD 11990.0
Etapa: Prospecto
Estado: Abierta



## 🔀 8. Orquestador de Agentes
Se ensambla el agente orquestador principal encapsulando los agentes especializados y la herramienta de registro dentro de funciones `@tool`, añadiendo memoria persistente con `InMemorySaver`. Exponemos cada capacidad como una **tool** y construimos el orquestador con `create_agent` `(LangChain 1.x)`. El orquestador **decide y enruta** a cualquiera de los cinco agentes:

- `consultar_catalogo` → 📗 Agente Catálogo y Precios
- `consultar_politicas` → ⚖️ Agente de politicas comerciales
- `consultar_crm` → 💸 Agente de proceso de ventas y crm
- `analizar_producto_tool` → 🖼️ Agente Multimodal
- `registrar_oportunidad` → ✍️ Agente de Accion

Ademas le damos **memoria** (`InMemorySaver` + `thread_id`), asi el orquestador recuerda la conversacion: si al registrar le falta un dato, lo pide y, cuando se lo das en el siguiente mensaje, **completa el registro**.


In [16]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.tools import tool
import uuid

@tool
def consultar_catalogo(pregunta: str) -> str:
    """Responde preguntas sobre productos, precios, disponibilidad y características."""
    return responder_catalogo(pregunta)

@tool
def consultar_politicas(pregunta: str) -> str:
    """Responde preguntas sobre descuentos, crédito, garantías, devoluciones y anticipos."""
    return responder_politicas(pregunta)
    
@tool
def consultar_crm(pregunta: str) -> str:
    """Responde preguntas sobre etapas del embudo, CRM, cierre de ventas y posventa."""
    return responder_crm(pregunta)

@tool
def analizar_producto_tool(ruta_imagen: str) -> str:
    """Analiza una imagen de un producto o ficha técnica."""
    return analizar_producto(ruta_imagen)

tools_orquestador = [consultar_catalogo, consultar_politicas, consultar_crm, registrar_oportunidad, analizar_producto_tool]

SYSTEM_PROMPT = """Eres el orquestador de PATITO S.A. Coordinas agentes especializados construidos con LangChain.

Agentes disponibles:
- consultar_catalogo: productos, precios, disponibilidad y características.
- consultar_politicas: descuentos, crédito, garantías, devoluciones y anticipos.
- consultar_crm: etapas del embudo, registro en CRM, cierre y posventa.
- registrar_oportunidad: registrar o guardar una oportunidad comercial en el archivo de texto.
- analizar_producto_tool: Analiza una imagen de producto o ficha técnica.

Reglas de ruteo:
- Si la consulta es sobre precios, productos o disponibilidad, usa consultar_catalogo.
- Si la consulta es sobre descuentos, crédito, garantías, devoluciones o anticipos, usa consultar_politicas.
- Si la consulta es sobre etapas, CRM, cierre o posventa, usa consultar_crm.
- Si el usuario pide registrar, guardar o crear una oportunidad, usa registrar_oportunidad.
- Si faltan datos obligatorios para registrar, pide los datos faltantes y no registres todavía.
- Si para responder completamente una consulta necesitas información de varias áreas, llama todas las herramientas necesarias antes de elaborar la respuesta final.
- No inventes información.
- Si no hay información suficiente, dilo explícitamente.
- No reveles el funcionamiento interno del sistema. No menciones herramientas (tools), agentes, retrievers, bases vectoriales, prompts ni procesos internos. El usuario solo debe recibir la respuesta final.
- Si se agregara una tool multimodal, usa esa tool únicamente cuando el usuario entregue una imagen o una ruta de imagen.
"""
memoria = InMemorySaver()

orquestador = create_agent(
    model=llm,
    tools=tools_orquestador,
    system_prompt=SYSTEM_PROMPT,
    checkpointer=memoria,
)

print("Tools registradas en el orquestador:")
for t in tools_orquestador:
    print("  -", t.name)

def _imprimir_pasos(resultado):
    for m in resultado["messages"]:
        for tc in (getattr(m, "tool_calls", None) or []):
            print(f"[TOOL] {tc['name']}({tc['args']})")
        if m.__class__.__name__ == "ToolMessage":
            print(f"[RESPONSE] {str(m.content)[:300]}\n")

def extraer_texto(content):
    """Devuelve solo texto plano si Gemini retorna bloques en lista."""
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        partes = []
        for b in content:
            if isinstance(b, dict):
                partes.append(b.get("text", ""))
            elif isinstance(b, str):
                partes.append(b)
        return "".join(partes).strip()
    return str(content)

def consultar(pregunta: str, thread_id: str = None):
    """Invoca al orquestador y muestra tools + respuesta final."""
    thread_id = thread_id or f"patito-{uuid.uuid4().hex[:8]}"
    config = {"configurable": {"thread_id": thread_id}}

    print(f">>> Usuario: {pregunta}\n")
    resultado = orquestador.invoke({"messages": [{"role": "user", "content": pregunta}]},config)
    _imprimir_pasos(resultado)
    print("=== Respuesta final ===")
    print(extraer_texto(resultado["messages"][-1].content))
    return resultado

Tools registradas en el orquestador:
  - consultar_catalogo
  - consultar_politicas
  - consultar_crm
  - registrar_oportunidad
  - analizar_producto_tool


## 🧪 9.  Pruebas y validaciones

Se ejecutan invocaciones de prueba sobre cada uno de los dominios **(precios, políticas, CRM)** para verificar que el orquestador llame a las tools correspondientes y muestre la secuencia interna de invocaciones en consola (`[TOOL] -> [RESPONSE]`).

### 9.1. Prueba - (Agente Catálogo y Precios)
Se ejecuta una consulta sobre precios de productos para validar que el orquestador identifique la intención y delegue la respuesta a la herramienta del Catálogo.

In [17]:
consultar("¿Cuál es el precio del Patito Pro 2026?")

>>> Usuario: ¿Cuál es el precio del Patito Pro 2026?

[TOOL] consultar_catalogo({'pregunta': '¿Cuál es el precio del Patito Pro 2026?'})
[RESPONSE] [{'type': 'text', 'text': 'El precio del Patito Pro 2026 es USD 1,299.', 'extras': {'signature': 'EjQKMgERTTIPI5m4ROrpXNA6jgxKcbJ2WnBVpuq2AaSgWdHkh4UjkjEvtsgHeVwp7N0msWKl'}}]

=== Respuesta final ===
El precio del Patito Pro 2026 es de USD 1,299.


{'messages': [HumanMessage(content='¿Cuál es el precio del Patito Pro 2026?', additional_kwargs={}, response_metadata={}, id='d8867beb-4954-4c58-8924-00af093346ca'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'consultar_catalogo', 'arguments': '{"pregunta": "\\u00bfCu\\u00e1l es el precio del Patito Pro 2026?"}'}, '__gemini_function_call_thought_signatures__': {'wHnZDpHA': 'EjQKMgERTTIPtA0TT22USn5yNuUKPaXJn9J6pcBy9u9yyI7Nxz0wOf91ODsGaBHPkKz76Q0Z'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019fa178-db19-76e3-a65e-64e5f05d4f9b-0', tool_calls=[{'name': 'consultar_catalogo', 'args': {'pregunta': '¿Cuál es el precio del Patito Pro 2026?'}, 'id': 'wHnZDpHA', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 806, 'output_tokens': 33, 'total_tokens': 839, 'input_token_details': {'cache_read': 0}}),
  ToolMessage(content=[{'type': 

### 9.2. Prueba - (Agente de de Políticas Comerciales)
Se envía una consulta sobre condiciones comerciales para comprobar que el orquestador redirija el flujo hacia el agente de Políticas.

In [18]:
consultar("¿Cuál es el descuento máximo que puede autorizar un vendedor?")

>>> Usuario: ¿Cuál es el descuento máximo que puede autorizar un vendedor?

[TOOL] consultar_politicas({'pregunta': '¿Cuál es el descuento máximo que puede autorizar un vendedor?'})
[RESPONSE] [{'type': 'text', 'text': 'El descuento máximo que un vendedor puede autorizar directamente, sin necesidad de aprobación adicional, es del 10%.', 'extras': {'signature': 'EjQKMgERTTIP8gNcJXF2ZLOwvcrfTtKaCxtCMp3qMxif6cCB1YfgfKxg+ItlaxK0LUB0jZfq'}}]

=== Respuesta final ===
El descuento máximo que un vendedor puede autorizar directamente, sin necesidad de aprobación adicional, es del 10%.


{'messages': [HumanMessage(content='¿Cuál es el descuento máximo que puede autorizar un vendedor?', additional_kwargs={}, response_metadata={}, id='ee8de9e7-7978-424f-ab42-3011cc715dee'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'consultar_politicas', 'arguments': '{"pregunta": "\\u00bfCu\\u00e1l es el descuento m\\u00e1ximo que puede autorizar un vendedor?"}'}, '__gemini_function_call_thought_signatures__': {'PvzJgH0I': 'EjQKMgERTTIPEXalhA0nBSbL/Mdo0XF3nqH4aY2nQA81r3N23IGx5HztbFVOi6NTc9yVoJGL'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019fa178-eb3f-76c2-b863-7b597f530c34-0', tool_calls=[{'name': 'consultar_politicas', 'args': {'pregunta': '¿Cuál es el descuento máximo que puede autorizar un vendedor?'}, 'id': 'PvzJgH0I', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 804, 'output_tokens': 31, 'total_tokens': 835, 'i

### 9.3. Prueba - (Agente de Proceso de Ventas y CRM)
Se realiza una pregunta sobre las etapas de venta para evaluar la correcta delegación hacia el agente experto en CRM.

In [19]:
consultar("¿Qué requisitos se necesitan para marcar una oportunidad como ganada?")

>>> Usuario: ¿Qué requisitos se necesitan para marcar una oportunidad como ganada?

[TOOL] consultar_crm({'pregunta': '¿Qué requisitos se necesitan para marcar una oportunidad como ganada?'})
[RESPONSE] [{'type': 'text', 'text': 'Para marcar una oportunidad como "ganada" en el CRM, es obligatorio registrar los siguientes requisitos:\n\n*   Orden de compra o contrato firmado por el cliente.\n*   Datos de facturación completos del cliente.\n*   Productos, cantidades y precios finales (incluyendo el d

=== Respuesta final ===
Para marcar una oportunidad como "ganada" en nuestro sistema, es necesario contar con la siguiente información y documentación:

*   **Orden de compra o contrato:** Debe estar debidamente firmado por el cliente.
*   **Datos de facturación:** La información completa del cliente para la emisión del comprobante.
*   **Detalle de la venta:** Productos, cantidades y precios finales (incluyendo el descuento aplicado y su respectiva autorización, si corresponde).
*   **Cond

{'messages': [HumanMessage(content='¿Qué requisitos se necesitan para marcar una oportunidad como ganada?', additional_kwargs={}, response_metadata={}, id='178f4e82-ab21-463b-b8ab-3d0fed324900'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'consultar_crm', 'arguments': '{"pregunta": "\\u00bfQu\\u00e9 requisitos se necesitan para marcar una oportunidad como ganada?"}'}, '__gemini_function_call_thought_signatures__': {'xLi5hmEN': 'EjQKMgERTTIPRjnFLkd27bOr3E9EkLiOZCIoGHYEkrvw0Baai8ngmks2bK1Bb6VUSfYZdYd4'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019fa179-0a55-7f92-b970-935a1d54abdc-0', tool_calls=[{'name': 'consultar_crm', 'args': {'pregunta': '¿Qué requisitos se necesitan para marcar una oportunidad como ganada?'}, 'id': 'xLi5hmEN', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 804, 'output_tokens': 30, 'total_tokens': 8

### 9.4. Ver el registro generado
Se abre y lee el archivo de texto `registro_oportunidades.txt` para asegurar que los eventos de escritura en disco se hayan guardado con el formato correcto.

In [20]:
print(Path(REGISTRO_PATH).read_text(encoding="utf-8") if Path(REGISTRO_PATH).exists() else "Aun no hay registros.")

ID: OP-0001
Fecha: 2026-07-25 22:31:27
Cliente: Comercial ABC
Producto: Patito Pro 2026
Cantidad: 10
Monto estimado: USD 11990.0
Etapa: Prospecto
Estado: Abierta
------------------------------
ID: OP-0010
Fecha: 2026-07-26 15:56:45
Cliente: Comercial ABC
Producto: Patito Pro 2026
Cantidad: 10
Monto estimado: USD 11990.0
Etapa: Prospecto
Estado: Abierta
------------------------------
ID: OP-0019
Fecha: 2026-07-26 17:53:47
Cliente: Comercial ABC
Producto: Patito Pro 2026
Cantidad: 10
Monto estimado: USD 11990.0
Etapa: Prospecto
Estado: Abierta
------------------------------
ID: OP-0028
Fecha: 2026-07-26 21:47:48
Cliente: Comercial ABC
Producto: Patito Pro 2026
Cantidad: 10
Monto estimado: USD 11990.0
Etapa: Prospecto
Estado: Abierta
------------------------------



## 📜 10. Generación de trazas (Logs de Ejecución)

Demuestra la trazabilidad interna del orquestador mediante la función consultar(), exponiendo el flujo completo de ejecución en tiempo real:

- Invocación de Herramientas (`[TOOL]`): Captura el enrutamiento automático hacia las tools correspondientes (**consultar_catalogo**, **consultar_politicas**, **consultar_crm** y  **registrar_oportunidad**).

- Respuesta de Herramientas (`[RESPONSE]`): Muestra los datos recuperados de las bases vectoriales antes de ser procesados por el modelo.

- Metadatos y Tokens: Registra el consumo exacto de tokens `(usage_metadata)`, identificadores de ejecuciones y firmas de seguridad de Gemini `(gemini-3.1-flash-lite)`.

>💬 Estructura de Mensajes: Retorna el historial formateado en objetos nativos de LangChain (`HumanMessage`, `AIMessage, ToolMessage`).

In [21]:
consultar("¿Cuál es el precio del Patito Pro 2026?")

>>> Usuario: ¿Cuál es el precio del Patito Pro 2026?

[TOOL] consultar_catalogo({'pregunta': '¿Cuál es el precio del Patito Pro 2026?'})
[RESPONSE] [{'type': 'text', 'text': 'El precio del Patito Pro 2026 es USD 1,299.', 'extras': {'signature': 'EjQKMgERTTIPEw946tAoC1WRe6u8VuQhWnd95PKI8pXhCj2txGocZK8XxL9/t1h/b4bHWGkR'}}]

=== Respuesta final ===
El precio del Patito Pro 2026 es de USD 1,299.


{'messages': [HumanMessage(content='¿Cuál es el precio del Patito Pro 2026?', additional_kwargs={}, response_metadata={}, id='61feac27-5d81-4059-a70b-a82a9dfa4753'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'consultar_catalogo', 'arguments': '{"pregunta": "\\u00bfCu\\u00e1l es el precio del Patito Pro 2026?"}'}, '__gemini_function_call_thought_signatures__': {'PRLyia64': 'EjQKMgERTTIPiEUe/2dmyIVA/0VEFc/Gl87SB3Yyz719/AD+k6Ftgs9exoyK/cyMFhn9Cti4'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019fa17a-cbbb-7821-9ffe-44c6f53a7a01-0', tool_calls=[{'name': 'consultar_catalogo', 'args': {'pregunta': '¿Cuál es el precio del Patito Pro 2026?'}, 'id': 'PRLyia64', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 806, 'output_tokens': 33, 'total_tokens': 839, 'input_token_details': {'cache_read': 0}}),
  ToolMessage(content=[{'type': 

In [22]:
consultar("¿Qué descuento puede autorizar un vendedor?")

>>> Usuario: ¿Qué descuento puede autorizar un vendedor?

[TOOL] consultar_politicas({'pregunta': '¿Qué descuento puede autorizar un vendedor?'})
[RESPONSE] [{'type': 'text', 'text': 'Un vendedor puede autorizar directamente descuentos de hasta el 10%, sin necesidad de aprobación adicional.', 'extras': {'signature': 'EjQKMgERTTIPa9CvdnUwPm5UPXq530QMXYfqz3FbToFRZuVvxNHjX+KRmL965HE4ca7ECHfF'}}]

=== Respuesta final ===
Un vendedor puede autorizar directamente descuentos de hasta el 10% sin necesidad de aprobación adicional.


{'messages': [HumanMessage(content='¿Qué descuento puede autorizar un vendedor?', additional_kwargs={}, response_metadata={}, id='865c3608-dc65-40c3-b9db-0808054219a3'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'consultar_politicas', 'arguments': '{"pregunta": "\\u00bfQu\\u00e9 descuento puede autorizar un vendedor?"}'}, '__gemini_function_call_thought_signatures__': {'DVHhHc1A': 'EjQKMgERTTIPDG6zKYya4BzSb/lNoaVL9sv/0yTimCbRnOwSJIkWMniDgKFtHeoX5KPnbNIu'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019fa17a-d954-71e2-aefa-6e3936279556-0', tool_calls=[{'name': 'consultar_politicas', 'args': {'pregunta': '¿Qué descuento puede autorizar un vendedor?'}, 'id': 'DVHhHc1A', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 800, 'output_tokens': 27, 'total_tokens': 827, 'input_token_details': {'cache_read': 0}}),
  ToolMessage(cont

In [23]:
consultar("¿Qué requisitos se necesitan para marcar una oportunidad como ganada?")

>>> Usuario: ¿Qué requisitos se necesitan para marcar una oportunidad como ganada?

[TOOL] consultar_crm({'pregunta': '¿Qué requisitos se necesitan para marcar una oportunidad como ganada?'})
[RESPONSE] [{'type': 'text', 'text': 'Para marcar una oportunidad como "ganada" en el CRM, es obligatorio registrar los siguientes requisitos:\n\n*   Orden de compra o contrato firmado por el cliente.\n*   Datos de facturación completos del cliente.\n*   Productos, cantidades y precios finales (incluyendo el d

=== Respuesta final ===
Para marcar una oportunidad como "ganada" en nuestro sistema, es necesario contar con la siguiente información y documentación:

*   **Orden de compra o contrato:** Debe estar debidamente firmado por el cliente.
*   **Datos de facturación:** Deben estar completos y validados.
*   **Detalle de la venta:** Productos, cantidades y precios finales (incluyendo cualquier descuento aplicado y su respectiva autorización).
*   **Condiciones comerciales:** Definir si la venta 

{'messages': [HumanMessage(content='¿Qué requisitos se necesitan para marcar una oportunidad como ganada?', additional_kwargs={}, response_metadata={}, id='4c1a1dc8-ebf4-4d25-9904-74a9735fce1a'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'consultar_crm', 'arguments': '{"pregunta": "\\u00bfQu\\u00e9 requisitos se necesitan para marcar una oportunidad como ganada?"}'}, '__gemini_function_call_thought_signatures__': {'iSWGVaeA': 'EjQKMgERTTIPTjtlph9slkgDwVXQcgQ+GIeW41chBtLACKtM0YePnSqBlCfGZbLhJ2AeOJPn'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019fa17a-ecce-72d2-9252-a1eb6930dbd8-0', tool_calls=[{'name': 'consultar_crm', 'args': {'pregunta': '¿Qué requisitos se necesitan para marcar una oportunidad como ganada?'}, 'id': 'iSWGVaeA', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 804, 'output_tokens': 30, 'total_tokens': 8

In [24]:
consultar("Registrar una oportunidad para Comercial ABC, 10 unidades de Patito Pro 2026, 8% de descuento, pago de contado.")

>>> Usuario: Registrar una oportunidad para Comercial ABC, 10 unidades de Patito Pro 2026, 8% de descuento, pago de contado.

[TOOL] consultar_catalogo({'pregunta': '¿Cuál es el precio unitario del producto Patito Pro 2026?'})
[RESPONSE] [{'type': 'text', 'text': 'El precio del Patito Pro 2026 es USD 1,299.', 'extras': {'signature': 'EjQKMgERTTIPh8YP6MsmzcrsXNqBXQxJjZMGill2ebTQFvCcPCP0t2lYMXo9djudzHdt/yhZ'}}]

[TOOL] registrar_oportunidad({'condicion_pago': 'contado', 'precio_unitario': 1299, 'confirmar': False, 'producto': 'Patito Pro 2026', 'descuento_aplicado': 8, 'monto_total': 11950.8, 'cliente': 'Comercial ABC', 'cantidad': 10})
[RESPONSE] No se registró la oportunidad. Faltan datos obligatorios: contacto.

=== Respuesta final ===
Para proceder con el registro de la oportunidad para Comercial ABC, necesito que me proporciones el nombre del **contacto** en dicha empresa.

Aquí tienes el resumen de la información que tengo hasta ahora:
*   **Producto:** Patito Pro 2026
*   **Cantid

{'messages': [HumanMessage(content='Registrar una oportunidad para Comercial ABC, 10 unidades de Patito Pro 2026, 8% de descuento, pago de contado.', additional_kwargs={}, response_metadata={}, id='187d4d76-3ac5-4296-ad34-8987958fd50a'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'consultar_catalogo', 'arguments': '{"pregunta": "\\u00bfCu\\u00e1l es el precio unitario del producto Patito Pro 2026?"}'}, '__gemini_function_call_thought_signatures__': {'Gx0L4lat': 'EjQKMgERTTIP/6TTRhAqBajXvHA0g92t7sszR1xn1EfpEpyMrgo8pgfR+p4FBM892fMqqVn3'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019fa17b-007c-7992-8e71-67776f798069-0', tool_calls=[{'name': 'consultar_catalogo', 'args': {'pregunta': '¿Cuál es el precio unitario del producto Patito Pro 2026?'}, 'id': 'Gx0L4lat', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 823, 'output_to

## 💭 11. Interfaz del Asistente (JupyterLab + `ipywidgets`)

La construcción de un **Chatbot interactivo** en JupyterLab permite interactuar de forma fluida con el orquestador multiagente de **Patito S.A.** A través del chat, las consultas se dirigen a los agentes especializados de **catálogo**, **políticas comerciales** y **CRM**, quienes ejecutan **herramientas con RAG** sobre las bases de conocimiento para generar respuestas contextualizadas.

- Componentes UI: Utiliza áreas de texto, botones primarios y un renderizador `HTML` con `CSS` adaptativo que genera burbujas de chat diferenciadas para usuario y asistente.

- Ejecución No Bloqueante (`threading`): Procesa las solicitudes en un hilo secundario para evitar congelar el cuaderno mientras el modelo responde.

- Gestión de Estado: Incluye historial de conversación persistente, manejo de errores y sanitización de datos.

In [25]:
import threading
import uuid
import traceback
import html as html_lib
import ipywidgets as widgets
from IPython.display import display

# CONFIGURACIÓN DE MEMORIA
config_patito = {
    "configurable": {
        "thread_id": str(uuid.uuid4())
    }
}

historial_patito = []

out_debug = widgets.Output()

# CSS

CSS_CHAT = """

<style>

.patito-card {
    background:white;
    border-radius:12px;
    padding:16px;
    font-family:Arial, sans-serif;
    box-shadow:0 2px 10px rgba(0,0,0,0.1);
    max-width:700px;
    margin:auto;
}

.patito-header {
    background:#162447;
    color:white;
    padding:14px;
    border-radius:10px 10px 0 0;
    font-weight:bold;
}

.patito-chat-box {
    height:420px;
    overflow-y:auto;
    background:#fafafa;
    border:1px solid #ddd;
    border-radius:8px;
    padding:12px;
    display:flex;
    flex-direction:column-reverse;
}

.patito-row {
    display:flex;
    margin:8px 0;

}
.patito-row.user {
    justify-content:flex-end;
}

.patito-row.agent {
    justify-content:flex-start;
}

.patito-msg {
    max-width:80%;
    padding:10px 14px;
    border-radius:14px;
    line-height:1.4;
}

.user .patito-msg {
    background:#162447;
    color:white;
}

.agent .patito-msg {
    background:white;
    border:1px solid #ddd;
    color:#333;
}

.patito-vacio {
    text-align:center;
    color:#999;
    padding:60px;
}

button {
    border-radius:6px !important;
}

</style>

"""
# WIDGETS

box_chat = widgets.HTML(
    value="""
    <div class="patito-chat-box">
        <div class="patito-vacio">
            🤖 Asistente Patito S.A. listo.
        </div>
    </div>
    """
)

txt_input = widgets.Textarea(
    placeholder="Escribe tu petición...",
    layout=widgets.Layout(
        width="100%",
        height="60px"
    )
)

btn_enviar = widgets.Button(
    description="Enviar",
    icon="paper-plane",
    button_style="primary",
    layout=widgets.Layout(
        width="100%"
    )
)

btn_nuevo = widgets.Button(
    description="Nuevo chat",
    icon="refresh",
    layout=widgets.Layout(
        width="100%"
    )
)

lbl_estado = widgets.HTML(value="")

# RENDER DEL CHAT
def render_mensajes():
    if not historial_patito:
        return """
        <div class="patito-chat-box">
            <div class="patito-vacio">
            🤖 Asistente Patito S.A. listo.
            </div>
        </div>
        """

    html = [ '<div class="patito-chat-box">' ]
    for mensaje in reversed(historial_patito):
        tipo = mensaje["rol"]
        nombre = (
            "👤 Tú"
            if tipo == "user"
            else "🤖 Asistente"
        )

        texto = (html_lib.escape(mensaje["texto"]).replace("\n","<br>"))

        html.append(f"""
            <div class="patito-row {tipo}">
                <div class="patito-msg">
                    <b>{nombre}</b>
                    <br>
                    {texto}
                </div>
            </div>
            """
        )
    html.append("</div>")

    return "".join(html)
    
# CONSULTAR ORQUESTADOR
def ejecutar_orquestador(pregunta):
    
    global config_patito
    try:
        resultado = orquestador.invoke(
            {
                "messages":[
                    {
                        "role":"user",
                        "content":pregunta
                    }
                ]
            },
            config_patito
        )

        respuesta = extraer_texto(
            resultado["messages"][-1].content
        )

        herramientas = []
        
        for mensaje in resultado["messages"]:
            for tool in (
                getattr(
                    mensaje,
                    "tool_calls",
                    []
                )
                or []
            ):
                herramientas.append(
                    tool["name"]
                )

        historial_patito.append(
            {
                "rol":"agent",
                "texto":respuesta
            }
        )

    except Exception:
        historial_patito.append(
            {
                "rol":"agent",
                "texto":
                "⚠️ Error consultando el orquestador."
            }
        )
        with out_debug:
            traceback.print_exc()
    finally:
        
        box_chat.value = render_mensajes()
        lbl_estado.value = ""
        btn_enviar.disabled = False
        
# BOTÓN ENVIAR
def enviar_mensaje(_):
    pregunta = txt_input.value.strip()
    if not pregunta:
        return
    if btn_enviar.disabled:
        return
        
    btn_enviar.disabled = True
    txt_input.value = ""
    historial_patito.append(
        {
            "rol":"user",
            "texto":pregunta
        }
    )
    box_chat.value = render_mensajes()
    lbl_estado.value = """

    <small>
    ⏳ El asistente está pensando...
    </small>

    """
    hilo = threading.Thread(
        target=ejecutar_orquestador,
        args=(pregunta,)
    )
    hilo.start()
btn_enviar.on_click(enviar_mensaje)

# NUEVO CHAT
def nuevo_chat(_):
    global config_patito
    historial_patito.clear()
    config_patito = {
        "configurable":{
            "thread_id":
            str(uuid.uuid4())
        }
    }
    box_chat.value = render_mensajes()
btn_nuevo.on_click(nuevo_chat)

# DESPLIEGUE
panel_chat = widgets.VBox(
    [
        widgets.HTML(
            """
            <div class="patito-card">
            <div class="patito-header">
            🤖 Asistente IA del departamento de ventas - Patito S.A.
            </div>
            """
        ),
        box_chat,
        lbl_estado,
        txt_input,
        btn_enviar,
        btn_nuevo,
        out_debug,
        widgets.HTML("</div>")
    ],

    layout=widgets.Layout(
        width="100%"
    )
)

display(
    widgets.HTML(CSS_CHAT)
)
display(panel_chat)

HTML(value='\n\n<style>\n\n.patito-card {\n    background:white;\n    border-radius:12px;\n    padding:16px;\n…

### 🛠️ Aspectos clave de la implementación

**1.  Ejecución no bloqueante (`threading.Thread`)**: Al presionar el botón de envío, el procesamiento con el modelo se traslada a un hilo secundario mediante `_process_message_async`. Esto evita que el kernel de JupyterLab se congele y permite mantener animaciones de carga fluidas.

**2.  Estilos CSS variables (`var(--jp-...)`)**: Los colores de las burbujas y del contenedor utilizan variables de entorno de JupyterLab, lo que garantiza compatibilidad tanto en Modo Claro (Light) como en Modo Oscuro (Dark).

**3.  Manejo de errores y sanitización**: Los textos ingresados por el usuario y devueltos por el bot pasan por `html.escape()` antes de insertarse en el DOM, previniendo fallos en la renderización HTML o vulnerabilidades XSS en el entorno del cuaderno.

**4.  Resistencia de estado (`thread_id`)**: Mantiene la referencia del hilo dentro de **LangGraph/LangChain** para que las consultas subsecuentes recuerden el contexto de los mensajes previos de la sesión.

## ⚠️ 12. Listado de riesgos y mejoras futuras

| Categoría / Riesgo | Clasificación | Impacto / Efecto | Propuesta de Mejora Futura |
|-----------------------|------------------|---------------------|-------------------------------|
| **Fallo en Hilo Secundario** (`threading`) | Interfaz / UI | Si el hilo secundario falla, la interfaz puede quedar congelada en *"Consultando agentes..."* sin alertar al usuario. | Implementar un mecanismo de *timeout* y un bloque `try/except/finally` que notifique los fallos y restablezca el estado de la interfaz tras **15 segundos**. |
| **Escalabilidad de ChromaDB** | Infraestructura | Almacenar vectores de forma local funciona para prototipos, pero no escala con un alto volumen de documentos o consultas concurrentes. | Migrar el almacenamiento vectorial a un servicio gestionado en la nube, como **Pinecone**, **Qdrant** o un servidor dedicado de **Chroma**, con índices distribuidos. |
| **Aislamiento de Permisos (RBAC)** | Seguridad | Delegar la seguridad únicamente al nivel del *prompt* o de las colecciones puede resultar vulnerable a ataques de *prompt injection*. | Implementar un *middleware* de autenticación con control de acceso basado en roles (RBAC) que filtre las *tools* y los *retrievers* según el perfil del usuario. |
| **Escritura en Registro de Ventas** | Calidad de Datos | El agente podría registrar incorrectamente una oportunidad si la información proporcionada por el usuario es vaga o ambigua antes de la persistencia. | Incorporar validación estricta mediante esquemas de `Pydantic` y pruebas unitarias para las herramientas de acción antes de escribir los datos. |
| **Límites de Tasa de API (*Rate Limits*)** | Disponibilidad | Un volumen elevado de consultas puede agotar la cuota de la API de Google Gemini y dejar el chatbot temporalmente fuera de servicio. | Integrar una capa de caché semántica para consultas frecuentes y un mecanismo de reintentos con *exponential backoff* cuando se alcancen los límites de tasa. |